# Stage 4b: Student KD Training (Proof of Concept)

**Goal:** Train the pruned student against pre-computed self-play teacher targets.
Verify that distillation improves over the raw pruned baseline.

**Run on:** Colab Free (T4) or Pro (A100)

**Workflow:**
1. Configure variant + loss function
2. Stage data + load pruned model
3. Pre-KD mimicking evaluation (baseline snapshot)
4. Train with knowledge distillation
5. Post-KD mimicking evaluation (comparison table)
6. Export final model

**Outputs saved to:** `eval/{exp_id}/kd/{variant}_{loss}/` and `checkpoints/{exp_id}/kd/{variant}_{loss}/`

## Cell 1: Session Startup

In [ ]:
# === SESSION STARTUP ===
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os

REPO = "https://github.com/<your-username>/moshilite.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Cell 2: Experiment Configuration

All experiment parameters in one place. Change `VARIANT_NAME` and `LOSS_CONFIG`
to run different ablations — all output paths update automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT CONFIGURATION — edit these to run different ablations    ║
# ╚══════════════════════════════════════════════════════════════════════╝

EXPERIMENT_ID  = "prune30_v1"                     # Stage 2 experiment ID
VARIANT_NAME   = "v1_scattered_nonuni"             # Which pruned variant to train
LOSS_CONFIG    = "L2"                              # Loss function: L1, L2, L3, L4, L5

# ── Training hyperparameters ──
LR             = 1e-4
BATCH_SIZE     = 4
GRAD_ACCUM     = 4       # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
NUM_EPOCHS     = 25
ALPHA          = 0.7     # soft logit KD weight
DELTA          = 0.3     # hard label CE weight
TEMPERATURE    = 3.0     # KD temperature

# ── Derived paths (all keyed by VARIANT_NAME + LOSS_CONFIG) ──
RUN_ID = f"{VARIANT_NAME}_{LOSS_CONFIG}"

GDRIVE_BASE    = "/content/drive/MyDrive/moshilite"
GDRIVE_DATA    = f"{GDRIVE_BASE}/self_play_data/conversations"
LOCAL_DATA     = "/content/staged/self_play"
WEIGHTS_DIR    = f"{GDRIVE_BASE}/upstream_weights/moshiko"
PRUNED_WEIGHTS = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}/{VARIANT_NAME}.pt"

CHECKPOINT_DIR = f"{GDRIVE_BASE}/checkpoints/{EXPERIMENT_ID}/kd/{RUN_ID}"
EVAL_DIR       = f"{GDRIVE_BASE}/eval/{EXPERIMENT_ID}/kd/{RUN_ID}"
EXPORT_PATH    = f"{GDRIVE_BASE}/models/{RUN_ID}_kd.pt"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print(f"Experiment:  {EXPERIMENT_ID}")
print(f"Variant:     {VARIANT_NAME}")
print(f"Loss config: {LOSS_CONFIG}")
print(f"Run ID:      {RUN_ID}")
print(f"")
print(f"Pruned weights: {PRUNED_WEIGHTS}")
print(f"Checkpoints:    {CHECKPOINT_DIR}")
print(f"Eval output:    {EVAL_DIR}")
print(f"Export path:    {EXPORT_PATH}")

## Cell 3: Stage Self-Play Data to Local Disk

In [ ]:
import shutil
from pathlib import Path

# Copy self-play data to local NVMe for fast training reads
src = Path(GDRIVE_DATA)
dst = Path(LOCAL_DATA)

if dst.exists():
    print(f"Local data already staged at {dst}")
else:
    print(f"Staging data from GDrive to local disk...")
    shutil.copytree(str(src), str(dst))

npz_count = len(list(dst.rglob("*.npz")))
total_mb = sum(f.stat().st_size for f in dst.rglob("*.npz")) / 1e6
print(f"Staged: {npz_count} conversations, {total_mb:.0f} MB")

## Cell 4: Load Pruned Student Model

In [ ]:
import torch
import json
from pathlib import Path
from moshi.models import loaders

# Detect dtype
gpu_name = torch.cuda.get_device_name(0).lower()
DTYPE = torch.bfloat16 if any(x in gpu_name for x in ["a100", "h100", "l4"]) else torch.float16
print(f"Using {DTYPE} on {gpu_name}")

print(f"Loading pruned student: {PRUNED_WEIGHTS}")
weights_path = Path(WEIGHTS_DIR)
config = None

if weights_path.exists():
    config_path = weights_path / "config.json"
    if config_path.exists():
        with open(config_path) as f:
            config = json.load(f)

# Load the base model architecture, then overlay pruned weights
student = loaders.get_moshi_lm(
    PRUNED_WEIGHTS, lm_kwargs=config, device="cuda", dtype=DTYPE
)
student.train()

# Enable gradient checkpointing if available
if hasattr(student.transformer, "set_checkpointing"):
    student.transformer.set_checkpointing(True)
    print("Gradient checkpointing enabled")

n_params = sum(p.numel() for p in student.parameters()) / 1e9
n_trainable = sum(p.numel() for p in student.parameters() if p.requires_grad) / 1e9
print(f"Student loaded: {n_params:.2f}B params ({n_trainable:.2f}B trainable)")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## Cell 4b: Pre-KD Mimicking Evaluation

Evaluate the pruned model **before** distillation training.
This gives a baseline snapshot to measure how much KD improves the student.

**Time:** ~30s on A100, ~1 min on T4

In [ ]:
import time, json
from moshilite.distillation.evaluator import MimickingEvaluator
from moshilite.data.dataset import get_self_play_dataloader

# Create evaluator (same hyperparams as training)
evaluator = MimickingEvaluator(
    alpha=ALPHA, delta=DELTA, temperature=TEMPERATURE,
    device="cuda", use_amp=True,
)

# Val loader (same deterministic split used during training)
val_loader = get_self_play_dataloader(
    LOCAL_DATA, split="val", batch_size=BATCH_SIZE, val_fraction=0.1,
)

print("\n" + "="*68)
print(f"  PRE-KD MIMICKING EVALUATION  [{VARIANT_NAME}]")
print("="*68)
student.eval()
t0 = time.perf_counter()
pre_kd_results = evaluator.evaluate(student, val_loader)
t1 = time.perf_counter()
student.train()

print(f"  Completed in {t1 - t0:.1f}s\n")
for k, v in pre_kd_results.items():
    print(f"    {k}: {v}")

# Save pre-KD eval
pre_kd_path = os.path.join(EVAL_DIR, "pre_kd_eval.json")
with open(pre_kd_path, "w") as f:
    json.dump(pre_kd_results, f, indent=2)
print(f"\nSaved to {pre_kd_path}")

# Quick health check
if pre_kd_results.get("text_perplexity", 0) > 10000:
    print("\n" + "!"*60)
    print("  WARNING: Text perplexity is extremely high (>10,000).")
    print("  This pruned variant may be too damaged for KD to recover.")
    print("  Consider trying a less aggressive variant.")
    print("!"*60)

## Cell 5: Run Training

In [ ]:
from moshilite.distillation.trainer import StudentTrainer

trainer = StudentTrainer(
    student_model=student,
    data_dir=LOCAL_DATA,
    checkpoint_dir=CHECKPOINT_DIR,
    # Training hyperparameters
    lr=LR,
    batch_size=BATCH_SIZE,
    gradient_accumulation=GRAD_ACCUM,
    alpha=ALPHA,
    delta=DELTA,
    temperature=TEMPERATURE,
    # Checkpointing
    checkpoint_every=200,
    val_every=100,
    # Hardware
    device="cuda",
    use_amp=True,
    # W&B (optional)
    wandb_project="moshilite",
    wandb_run_name=RUN_ID,
)

summary = trainer.train(num_epochs=NUM_EPOCHS)

## Cell 6: Load Best Checkpoint & Training Summary

In [ ]:
# Load best checkpoint for final evaluation
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / "checkpoint_best.pt"
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location="cuda")
    student.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best checkpoint (val_loss={ckpt['best_val_loss']:.4f})")
else:
    print("No best checkpoint found, using final model")

student.eval()

print(f"\nTraining Summary [{RUN_ID}]:")
print(f"   Total steps:     {summary['total_steps']}")
print(f"   Best val loss:   {summary['best_val_loss']:.4f}")
print(f"   Final train loss:{summary['final_train_loss']:.4f}")
print(f"   Training time:   {summary['elapsed_seconds'] / 60:.1f} min")

## Cell 6b: Post-KD Mimicking Evaluation

Evaluate the distilled student on the same held-out self-play validation split
and compare against the pre-KD baseline from Cell 4b.

**Time:** ~30s on A100, ~1 min on T4

In [ ]:
import time, json

print("\n" + "="*68)
print(f"  POST-KD MIMICKING EVALUATION  [{RUN_ID}]")
print("="*68)
t0 = time.perf_counter()
post_kd_results = evaluator.evaluate(student, val_loader)
t1 = time.perf_counter()
print(f"  Completed in {t1 - t0:.1f}s\n")
for k, v in post_kd_results.items():
    print(f"    {k}: {v}")

# Print comparison table: Post-KD vs Pre-KD
evaluator.print_comparison(post_kd_results, pre_kd_results)

# Save all evaluation results
eval_output = {
    "run_id": RUN_ID,
    "variant_name": VARIANT_NAME,
    "loss_config": LOSS_CONFIG,
    "hyperparameters": {
        "lr": LR, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "alpha": ALPHA, "delta": DELTA, "temperature": TEMPERATURE,
        "num_epochs": NUM_EPOCHS,
    },
    "pre_kd": pre_kd_results,
    "post_kd": post_kd_results,
    "training_summary": summary,
}

# Save post-KD eval
post_kd_path = os.path.join(EVAL_DIR, "post_kd_eval.json")
with open(post_kd_path, "w") as f:
    json.dump({"post_kd": post_kd_results}, f, indent=2, default=str)

# Save combined results
combined_path = os.path.join(EVAL_DIR, "full_eval_results.json")
with open(combined_path, "w") as f:
    json.dump(eval_output, f, indent=2, default=str)

# Save training summary separately
summary_path = os.path.join(EVAL_DIR, "training_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nAll results saved to {EVAL_DIR}/")
print(f"  - pre_kd_eval.json")
print(f"  - post_kd_eval.json")
print(f"  - full_eval_results.json")
print(f"  - training_summary.json")

## Cell 7: Export Student Model

In [ ]:
# Save the distilled student as a standalone model
from pathlib import Path

torch.save({
    "model_state_dict": student.state_dict(),
    "config": {
        "run_id": RUN_ID,
        "variant_name": VARIANT_NAME,
        "loss_config": LOSS_CONFIG,
        "best_val_loss": summary["best_val_loss"],
        "training_steps": summary["total_steps"],
        "num_epochs": NUM_EPOCHS,
        "alpha": ALPHA,
        "delta": DELTA,
        "temperature": TEMPERATURE,
    },
    "pre_kd_eval": pre_kd_results,
    "post_kd_eval": post_kd_results,
}, EXPORT_PATH)

# Update model manifest
try:
    from moshilite.utils.experiment import update_model_manifest
    update_model_manifest(
        model_filename=f"{RUN_ID}_kd.pt",
        source_checkpoint=str(best_ckpt),
        experiment_id=EXPERIMENT_ID,
        metrics={
            "best_val_loss": summary["best_val_loss"],
            "training_steps": summary["total_steps"],
            **post_kd_results,
        },
    )
except Exception as e:
    print(f"Manifest update failed (non-critical): {e}")

export_size = Path(EXPORT_PATH).stat().st_size / 1e9
print(f"Student exported to {EXPORT_PATH}")
print(f"Size: {export_size:.2f} GB")